## Hardware Check

In [ ]:
!nvidia-smi
import torch
print(f'\nPyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA device: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

## Setup Environment
Run this cell to clone the repository securely and install dependencies.

In [ ]:
import os
import getpass
from pathlib import Path

GITHUB_USER = 'sattary'
REPO_NAME = 'ali_proj'
BRANCH = 'fix-review'  # Update if on a different branch
PROJECT_DIR = 'ali_proj'

if not Path(PROJECT_DIR).exists():
    print('Enter your GitHub Personal Access Token (PAT):')
    PAT = getpass.getpass()
    REPO_URL = f'https://{PAT}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    print(f'Cloning {REPO_NAME} (branch: {BRANCH})...')
    os.system(f'git clone -b {BRANCH} {REPO_URL}')
else:
    print('Repository already cloned. Pulling latest changes...')
    os.system(f'cd {PROJECT_DIR} && git pull origin {BRANCH}')

os.chdir(PROJECT_DIR)

print('\nInstalling uv...')
os.system('pip install -q uv')

os.environ['MPLBACKEND'] = 'Agg'
print('\nSyncing dependencies...')
os.system('uv sync')
print('\n\u2713 Setup complete!')

## Phase 1: Generate Dataset
Generate the synthetic dataset to HDF5 shards. Adjust `NUM_SAMPLES` as needed.

In [ ]:
NUM_SAMPLES = 50000
SHARD_SIZE = 500
OUT_DIR = 'data/training_set'

!uv run phase-unwrap generate \
    --num-samples {NUM_SAMPLES} \
    --shard-size {SHARD_SIZE} \
    --out-dir {OUT_DIR} \
    --seed 1337

print('\n\u2713 Data generation complete!')

## Phase 2: Hyperparameter Optimization
Find the absolute best model configuration using Optuna.

In [ ]:
N_TRIALS = 30
TUNE_EPOCHS = 15
BATCH_SIZE = 32

!uv run phase-unwrap tune \
    --use-amp \
    --n-trials {N_TRIALS} \
    --tune-epochs {TUNE_EPOCHS} \
    --study-name cloud_hpo \
    --n-workers 1 \
    --batch-size {BATCH_SIZE} \
    --data-dir {OUT_DIR}

print('\n\u2713 Tuning complete! Best config is in runs/optuna/best_config.yaml')

## Next Steps
Download `runs/optuna/best_config.yaml` to persist it across environments, then run `02_train_and_evaluate.ipynb`.